# Piper Colab Training Demo

This notebook is optimized for Colab so you can change one configuration cell, then run all cells.


## 1. Configuration

Edit only this cell when you want to switch voice name, dataset location, or training settings.


In [ ]:
from pathlib import Path

# Core paths
PROJECT_ROOT = Path("/content/piper1-gpl")
VOICE_NAME = "my_voice"

# Dataset mode
USE_SAMPLE_DATASET = True
USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_DATA_DIR = "/content/drive/MyDrive/piper-datasets/my_voice"

# Sample dataset download (used only when USE_SAMPLE_DATASET=True)
# Priority: SAMPLE_DATASET_URL -> SAMPLE_DATASET_FILE_ID
SAMPLE_DATASET_URL = "https://github.com/izlabs/voice-desk-tts-assets/releases/download/v0.1.0/audio_train_demo.zip"
SAMPLE_DATASET_FILE_ID = ""
SAMPLE_DATASET_ARCHIVE = "audio_train_demo.zip"

# Data paths
DATA_DIR = Path(DRIVE_DATA_DIR) if USE_GOOGLE_DRIVE else PROJECT_ROOT / "wav"
CSV_PATH = DATA_DIR / "metadata.csv"
CACHE_DIR = PROJECT_ROOT / "cache" / VOICE_NAME
CONFIG_PATH = PROJECT_ROOT / f"{VOICE_NAME}.onnx.json"
OUTPUT_ONNX = Path(f"/content/{VOICE_NAME}.onnx")
OUTPUT_CONFIG = Path(f"/content/{VOICE_NAME}.onnx.json")
TEST_WAV = Path(f"/content/{VOICE_NAME}_test.wav")

# Training settings
ESPEAK_VOICE = "vi"
SAMPLE_RATE = 22050
BATCH_SIZE = 16
MAX_EPOCHS = 4000
USE_CUDA_FOR_INFERENCE = True
TEST_TEXT = "Xin chao, day la ban demo huan luyen Piper tren Google Colab."

# Base checkpoint
BASE_CKPT_URL = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/hfc_male/medium/epoch%3D2785-step%3D2128064.ckpt"
BASE_CONFIG_URL = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/hfc_male/medium/config.json?download=true"
RAW_BASE_CKPT = PROJECT_ROOT / "pretrained-model.ckpt"
CLEAN_BASE_CKPT = PROJECT_ROOT / "pretrained-model-cleaned.ckpt"

print({
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "VOICE_NAME": VOICE_NAME,
    "DATA_DIR": str(DATA_DIR),
    "CSV_PATH": str(CSV_PATH),
    "CONFIG_PATH": str(CONFIG_PATH),
    "CACHE_DIR": str(CACHE_DIR),
    "OUTPUT_ONNX": str(OUTPUT_ONNX),
    "OUTPUT_CONFIG": str(OUTPUT_CONFIG),
    "TEST_WAV": str(TEST_WAV),
})


## 2. Environment setup


In [ ]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install onnxscript gdown


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build espeak-ng espeak-ng-data libespeak-ng-dev pkg-config ffmpeg
!pkg-config --modversion espeak-ng


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
    print(f"Mounted Google Drive at {DRIVE_MOUNT_POINT}")
else:
    print("Google Drive disabled.")


In [ ]:
%cd /content
!rm -rf /content/piper1-gpl
!git clone https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install --upgrade pip setuptools wheel scikit-build cmake ninja
!python3 -m pip install -e ".[train]"
!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace -v


## 3. Dataset setup


In [ ]:
import shutil
from pathlib import Path

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if USE_SAMPLE_DATASET:
    %cd /content
    !rm -f "$SAMPLE_DATASET_ARCHIVE"
    !rm -rf "$DATA_DIR"
    if SAMPLE_DATASET_URL:
        !wget -O "$SAMPLE_DATASET_ARCHIVE" "$SAMPLE_DATASET_URL"
        print(f"Downloaded sample dataset from URL: {SAMPLE_DATASET_URL}")
    else:
        assert SAMPLE_DATASET_FILE_ID, "Set SAMPLE_DATASET_URL or SAMPLE_DATASET_FILE_ID when USE_SAMPLE_DATASET=True"
        !gdown "$SAMPLE_DATASET_FILE_ID" -O "$SAMPLE_DATASET_ARCHIVE"
        print(f"Downloaded sample dataset from Google Drive file id: {SAMPLE_DATASET_FILE_ID}")
    !unzip -o "$SAMPLE_DATASET_ARCHIVE" -d "$DATA_DIR"
    print(f"Sample dataset extracted to {DATA_DIR}")
else:
    print(f"Using custom dataset directory: {DATA_DIR}")

assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"
assert CSV_PATH.exists(), f"metadata.csv not found: {CSV_PATH}"
print(f"Dataset ready: {DATA_DIR}")
print(f"Metadata ready: {CSV_PATH}")


## 4. Download and clean base checkpoint


In [ ]:
%cd /content/piper1-gpl
!wget -O "$RAW_BASE_CKPT" "$BASE_CKPT_URL"
!wget -O /content/piper1-gpl/base-config.json "$BASE_CONFIG_URL"


In [ ]:
%cd /content/piper1-gpl/src
from piper.train.vits.lightning import VitsModel
import inspect
import logging
import torch
from pathlib import PosixPath, WindowsPath

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def convert_paths_to_strings(data):
    if isinstance(data, dict):
        return {key: convert_paths_to_strings(value) for key, value in data.items()}
    if isinstance(data, list):
        return [convert_paths_to_strings(item) for item in data]
    if isinstance(data, (PosixPath, WindowsPath)):
        return str(data)
    return data

def clean_and_save_checkpoint(input_path: str, output_path: str):
    checkpoint = torch.load(input_path, map_location='cpu', weights_only=False)
    cleaned_checkpoint = convert_paths_to_strings(checkpoint)

    if 'hyper_parameters' in cleaned_checkpoint:
        init_signature = inspect.signature(VitsModel.__init__)
        valid_params = set(init_signature.parameters.keys())
        checkpoint_params = set(cleaned_checkpoint['hyper_parameters'].keys())
        invalid_params = checkpoint_params - valid_params
        for param in sorted(list(invalid_params)):
            del cleaned_checkpoint['hyper_parameters'][param]

    torch.save(cleaned_checkpoint, output_path)
    print(f"Cleaned checkpoint saved to {output_path}")

clean_and_save_checkpoint(str(RAW_BASE_CKPT), str(CLEAN_BASE_CKPT))


In [ ]:
main_py_path = PROJECT_ROOT / "src" / "piper" / "train" / "__main__.py"
prepend_code = (
    "import torch\n"
    "import pathlib\n"
    "torch.serialization.add_safe_globals([pathlib.PosixPath])\n"
)

original_content = main_py_path.read_text()
if prepend_code not in original_content:
    main_py_path.write_text(prepend_code + original_content)
    print("Patched __main__.py with safe_globals support.")
else:
    print("__main__.py already patched.")


## 5. Train


In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Training voice: {VOICE_NAME}")
print(f"Using dataset: {DATA_DIR}")
print(f"Using metadata: {CSV_PATH}")
print(f"Writing config to: {CONFIG_PATH}")


In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train fit \
  --data.voice_name "$VOICE_NAME" \
  --data.csv_path "$CSV_PATH" \
  --data.audio_dir "$DATA_DIR" \
  --model.sample_rate "$SAMPLE_RATE" \
  --data.espeak_voice "$ESPEAK_VOICE" \
  --data.cache_dir "$CACHE_DIR" \
  --data.config_path "$CONFIG_PATH" \
  --data.batch_size "$BATCH_SIZE" \
  --ckpt_path "$CLEAN_BASE_CKPT" \
  --trainer.max_epochs "$MAX_EPOCHS"


## 6. Export latest checkpoint automatically


In [ ]:
from pathlib import Path

checkpoint_candidates = sorted(
    PROJECT_ROOT.glob("lightning_logs/**/checkpoints/*.ckpt"),
    key=lambda p: p.stat().st_mtime,
)
assert checkpoint_candidates, "No checkpoints found under lightning_logs/**/checkpoints"
LATEST_CKPT = checkpoint_candidates[-1]
print(f"Latest checkpoint: {LATEST_CKPT}")


In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train.export_onnx \
  --checkpoint "$LATEST_CKPT" \
  --output-file "$OUTPUT_ONNX"


In [ ]:
import shutil
shutil.copyfile(CONFIG_PATH, OUTPUT_CONFIG)
print(f"Exported model: {OUTPUT_ONNX}")
print(f"Exported config: {OUTPUT_CONFIG}")


## 7. Quick inference test


In [ ]:
%cd /content/piper1-gpl/src
import wave
from IPython.display import Audio, display
from piper import PiperVoice, SynthesisConfig

voice = PiperVoice.load(
    model_path=str(OUTPUT_ONNX),
    config_path=str(OUTPUT_CONFIG),
    use_cuda=USE_CUDA_FOR_INFERENCE,
)

syn_config = SynthesisConfig(
    volume=1.0,
    length_scale=1.0,
    noise_scale=1.0,
    noise_w_scale=1.0,
    normalize_audio=False,
)

with wave.open(str(TEST_WAV), "wb") as wav_file:
    voice.synthesize_wav(TEST_TEXT, wav_file, syn_config=syn_config)

print(f"Saved test audio to {TEST_WAV}")
display(Audio(str(TEST_WAV)))


## Notes

- To download a sample dataset from GitHub Releases or any public URL: set `USE_SAMPLE_DATASET = True` and fill `SAMPLE_DATASET_URL`.
- If `SAMPLE_DATASET_URL` is empty, the notebook falls back to `SAMPLE_DATASET_FILE_ID` through Google Drive.
- For your own dataset on Drive: set `USE_SAMPLE_DATASET = False`, `USE_GOOGLE_DRIVE = True`, and update `DRIVE_DATA_DIR`.
- Your dataset folder should contain audio files and `metadata.csv`.
- You usually only need to edit the configuration cell, then choose `Runtime > Run all` in Colab.
